# 1. Carregamento das bases processadas

Nesta etapa são carregadas as bases previamente preparadas e validadas das duas propriedades utilizadas no estudo: a propriedade localizada em Cerqueira César (SP) e a Fazenda Escola da UENP.

A preparação individual das fontes foi realizada anteriormente, incluindo tratamento de inconsistências, padronização, integração de informações e avaliação temporal das variáveis.

Neste notebook, as bases processadas foram utilizadas para a etapa de harmonização entre as propriedades e, posteriormente, para a construção e avaliação dos modelos de Aprendizado de Máquina.

Também são carregadas as tabelas de decisão das variáveis, que registram as escolhas metodológicas realizadas durante a preparação de cada propriedade.

In [ ]:
# ============================================================
# 1. CARREGAMENTO DAS BASES PROCESSADAS
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# LOCALIZAÇÃO DA PASTA DE DADOS PROCESSADOS
# ------------------------------------------------------------

caminhos_processados = [
    Path("../dados/processados"),
    Path("dados/processados")
]

pasta_processados = next(
    (
        caminho
        for caminho in caminhos_processados
        if caminho.exists()
    ),
    None
)

if pasta_processados is None:
    raise FileNotFoundError(
        "Não foi possível localizar a pasta dados/processados."
    )

print("Pasta utilizada:")
print(pasta_processados.resolve())


# ------------------------------------------------------------
# CARREGAMENTO DOS ARQUIVOS
# ------------------------------------------------------------

base_cc = pd.read_csv(
    pasta_processados / "cerqueira_cesar_base_processada.csv"
)

decisao_cc = pd.read_csv(
    pasta_processados / "cerqueira_cesar_decisao_variaveis.csv"
)

base_uenp = pd.read_csv(
    pasta_processados / "uenp_base_processada.csv"
)

decisao_uenp = pd.read_csv(
    pasta_processados / "uenp_decisao_variaveis.csv"
)


# ------------------------------------------------------------
# CONFERÊNCIA INICIAL
# ------------------------------------------------------------

print("\nBASES CARREGADAS")
print("-" * 60)

print(
    "Cerqueira César:",
    base_cc.shape
)

print(
    "UENP:",
    base_uenp.shape
)

print(
    "Decisões Cerqueira César:",
    decisao_cc.shape
)

print(
    "Decisões UENP:",
    decisao_uenp.shape
)

In [ ]:
# ============================================================
# 1.1 CONFERÊNCIA DAS ESTRUTURAS
# ============================================================

print("COLUNAS — CERQUEIRA CÉSAR")
print("-" * 60)

for i, coluna in enumerate(base_cc.columns, start=1):
    print(f"{i:02d}. {coluna}")


print("\nCOLUNAS — UENP")
print("-" * 60)

for i, coluna in enumerate(base_uenp.columns, start=1):
    print(f"{i:02d}. {coluna}")


print("\nPRIMEIROS REGISTROS — CERQUEIRA CÉSAR")

display(
    base_cc.head()
)


print("\nPRIMEIROS REGISTROS — UENP")

display(
    base_uenp.head()
)

# 2. Harmonização das variáveis entre as propriedades

As bases processadas das duas propriedades apresentam diferenças tanto na disponibilidade quanto na origem das informações.

Antes da integração, as variáveis foram comparadas considerando seu significado e sua forma de obtenção, evitando que campos com nomes semelhantes, mas sem equivalência metodológica, sejam tratados como uma mesma característica.

Algumas variáveis apresentam equivalência direta, como produção de leite, idade e mês do controle. Outras representam o mesmo conceito, porém foram obtidas de formas diferentes entre as propriedades. Esse é o caso dos dias em lactação e da produção anterior.

Também existem atributos disponíveis exclusivamente em uma das propriedades, como peso corporal na UENP e determinadas informações reprodutivas em Cerqueira César. Esses atributos foram mantidos nas bases processadas de origem; na base harmonizada, o peso corporal foi preservado, mas esses atributos exclusivos não foram utilizados nos modelos conjuntos.

A harmonização foi realizada inicialmente sobre o conjunto de variáveis semanticamente comparáveis entre as duas fontes.

In [ ]:
# ============================================================
# 2.1 PADRONIZAÇÃO DOS TIPOS DE DADOS
# ============================================================

cc_harm = base_cc.copy()
uenp_harm = base_uenp.copy()


# ------------------------------------------------------------
# DATAS
# ------------------------------------------------------------

cc_harm["data_prova"] = pd.to_datetime(
    cc_harm["data_prova"],
    errors="coerce"
)

uenp_harm["data_prova"] = pd.to_datetime(
    uenp_harm["data_prova"],
    errors="coerce"
)

if "data_pesagem" in uenp_harm.columns:
    uenp_harm["data_pesagem"] = pd.to_datetime(
        uenp_harm["data_pesagem"],
        errors="coerce"
    )


# ------------------------------------------------------------
# VARIÁVEIS NUMÉRICAS — CERQUEIRA CÉSAR
# ------------------------------------------------------------

numericas_cc = [
    "kg_leite",
    "del",
    "kg_anterior",
    "idade_anos",
    "dias_ate_parto_previsto",
    "mes_controle"
]

for coluna in numericas_cc:
    cc_harm[coluna] = pd.to_numeric(
        cc_harm[coluna],
        errors="coerce"
    )


# ------------------------------------------------------------
# VARIÁVEIS NUMÉRICAS — UENP
# ------------------------------------------------------------

numericas_uenp = [
    "ano_origem",
    "kg_leite",
    "peso_kg",
    "idade_anos",
    "del_derivado",
    "kg_leite_anterior",
    "dias_desde_controle_anterior",
    "mes_controle",
    "dias_desde_pesagem"
]

for coluna in numericas_uenp:
    uenp_harm[coluna] = pd.to_numeric(
        uenp_harm[coluna],
        errors="coerce"
    )


# ------------------------------------------------------------
# VALIDAÇÃO
# ------------------------------------------------------------

print("TIPOS APÓS PADRONIZAÇÃO")
print("-" * 60)

print("\nCerqueira César:")
display(
    cc_harm[
        [
            "data_prova",
            "kg_leite",
            "del",
            "kg_anterior",
            "idade_anos",
            "mes_controle"
        ]
    ].dtypes
)

print("\nUENP:")
display(
    uenp_harm[
        [
            "data_prova",
            "kg_leite",
            "peso_kg",
            "del_derivado",
            "kg_leite_anterior",
            "idade_anos",
            "mes_controle"
        ]
    ].dtypes
)

print("\nDatas inválidas:")
print(
    "Cerqueira César:",
    cc_harm["data_prova"].isna().sum()
)

print(
    "UENP:",
    uenp_harm["data_prova"].isna().sum()
)

In [ ]:
# ============================================================
# 2.2 MAPA DE EQUIVALÊNCIA DAS VARIÁVEIS
# ============================================================

mapa_harmonizacao = pd.DataFrame([
    {
        "conceito": "produção de leite",
        "cerqueira_cesar": "kg_leite",
        "uenp": "kg_leite",
        "equivalencia": "direta",
        "decisao": "harmonizar"
    },
    {
        "conceito": "data do controle",
        "cerqueira_cesar": "data_prova",
        "uenp": "data_prova",
        "equivalencia": "direta",
        "decisao": "harmonizar"
    },
    {
        "conceito": "raça",
        "cerqueira_cesar": "raca_padronizada",
        "uenp": "raca_padronizada",
        "equivalencia": "direta",
        "decisao": "harmonizar"
    },
    {
        "conceito": "idade",
        "cerqueira_cesar": "idade_anos",
        "uenp": "idade_anos",
        "equivalencia": "direta",
        "decisao": "harmonizar"
    },
    {
        "conceito": "mês do controle",
        "cerqueira_cesar": "mes_controle",
        "uenp": "mes_controle",
        "equivalencia": "direta",
        "decisao": "harmonizar"
    },
    {
        "conceito": "dias em lactação",
        "cerqueira_cesar": "del",
        "uenp": "del_derivado",
        "equivalencia": "conceitual",
        "decisao": "harmonizar com rastreabilidade"
    },
    {
        "conceito": "produção anterior",
        "cerqueira_cesar": "kg_anterior",
        "uenp": "kg_leite_anterior",
        "equivalencia": "conceitual",
        "decisao": "avaliar antes de harmonizar"
    },
    {
        "conceito": "peso corporal",
        "cerqueira_cesar": None,
        "uenp": "peso_kg",
        "equivalencia": "exclusiva UENP",
        "decisao": "não usar no modelo conjunto principal"
    },
    {
        "conceito": "estado reprodutivo",
        "cerqueira_cesar": "estado_reprodutivo_padronizado",
        "uenp": None,
        "equivalencia": "exclusiva CC",
        "decisao": "não usar no modelo conjunto principal"
    },
    {
        "conceito": "dias até parto previsto",
        "cerqueira_cesar": "dias_ate_parto_previsto",
        "uenp": None,
        "equivalencia": "exclusiva CC",
        "decisao": "não usar no modelo conjunto principal"
    },
    {
        "conceito": "tipo de parto",
        "cerqueira_cesar": "tipo_parto_validado",
        "uenp": None,
        "equivalencia": "exclusiva CC",
        "decisao": "não usar no modelo conjunto principal"
    }
])

display(mapa_harmonizacao)

In [ ]:
# ============================================================
# 2.3 COBERTURA DAS VARIÁVEIS COMUNS
# ============================================================

cc_alvo = cc_harm.loc[
    cc_harm["kg_leite"].notna()
].copy()

uenp_alvo = uenp_harm.loc[
    uenp_harm["kg_leite"].notna()
].copy()


variaveis_comuns = pd.DataFrame([
    {
        "variavel_harmonizada": "raca",
        "cc": "raca_padronizada",
        "uenp": "raca_padronizada"
    },
    {
        "variavel_harmonizada": "idade_anos",
        "cc": "idade_anos",
        "uenp": "idade_anos"
    },
    {
        "variavel_harmonizada": "del",
        "cc": "del",
        "uenp": "del_derivado"
    },
    {
        "variavel_harmonizada": "producao_anterior",
        "cc": "kg_anterior",
        "uenp": "kg_leite_anterior"
    },
    {
        "variavel_harmonizada": "mes_controle",
        "cc": "mes_controle",
        "uenp": "mes_controle"
    }
])


resultado_cobertura = []

for _, linha in variaveis_comuns.iterrows():

    coluna_cc = linha["cc"]
    coluna_uenp = linha["uenp"]

    n_cc = cc_alvo[coluna_cc].notna().sum()
    n_uenp = uenp_alvo[coluna_uenp].notna().sum()

    resultado_cobertura.append({
        "variavel": linha["variavel_harmonizada"],

        "CC_preenchidos": n_cc,
        "CC_total": len(cc_alvo),
        "CC_cobertura_%": round(
            n_cc / len(cc_alvo) * 100,
            2
        ),

        "UENP_preenchidos": n_uenp,
        "UENP_total": len(uenp_alvo),
        "UENP_cobertura_%": round(
            n_uenp / len(uenp_alvo) * 100,
            2
        ),

        "total_preenchidos": n_cc + n_uenp,
        "total_alvos": len(cc_alvo) + len(uenp_alvo),
        "cobertura_conjunta_%": round(
            (n_cc + n_uenp)
            / (len(cc_alvo) + len(uenp_alvo))
            * 100,
            2
        )
    })


cobertura_harmonizacao = pd.DataFrame(
    resultado_cobertura
)

print("PRODUÇÕES DISPONÍVEIS")
print("-" * 60)

print(
    "Cerqueira César:",
    len(cc_alvo)
)

print(
    "UENP:",
    len(uenp_alvo)
)

print(
    "Total:",
    len(cc_alvo) + len(uenp_alvo)
)

print("\nCOBERTURA DAS VARIÁVEIS COMUNS")

display(cobertura_harmonizacao)

### 2.4 Avaliação da equivalência da produção anterior

As duas propriedades possuem informações relacionadas à produção de leite anterior ao controle atual, porém essas variáveis apresentam diferenças em sua origem.

Em Cerqueira César, o valor de produção anterior é registrado diretamente na planilha de controle leiteiro. Na UENP, essa informação foi derivada a partir da produção mensurada no controle cronologicamente anterior do mesmo animal.

Embora ambas representem informação produtiva histórica disponível antes da produção atual, suas características foram comparadas antes da harmonização definitiva.

Foram avaliadas a cobertura, a ocorrência de valores iguais a zero e a distribuição dos valores disponíveis em cada propriedade.

In [ ]:
# ============================================================
# 2.4 AVALIAÇÃO DA PRODUÇÃO ANTERIOR
# ============================================================

print("PRODUÇÃO ANTERIOR — COMPARAÇÃO ENTRE PROPRIEDADES")
print("-" * 60)

# ------------------------------------------------------------
# CERQUEIRA CÉSAR
# ------------------------------------------------------------

anterior_cc = cc_alvo["kg_anterior"]

print("\nCERQUEIRA CÉSAR")
print("-" * 30)

print(
    "Preenchidos:",
    anterior_cc.notna().sum()
)

print(
    "Ausentes:",
    anterior_cc.isna().sum()
)

print(
    "Valores iguais a zero:",
    (anterior_cc == 0).sum()
)

print("\nDistribuição:")

display(
    anterior_cc.describe()
)


# ------------------------------------------------------------
# UENP
# ------------------------------------------------------------

anterior_uenp = uenp_alvo["kg_leite_anterior"]

print("\nUENP")
print("-" * 30)

print(
    "Preenchidos:",
    anterior_uenp.notna().sum()
)

print(
    "Ausentes:",
    anterior_uenp.isna().sum()
)

print(
    "Valores iguais a zero:",
    (anterior_uenp == 0).sum()
)

print("\nDistribuição:")

display(
    anterior_uenp.describe()
)


# ------------------------------------------------------------
# UENP — MOTIVO DA AUSÊNCIA
# ------------------------------------------------------------

print("\nUENP — SITUAÇÃO ANTERIOR QUANDO kg_leite_anterior ESTÁ AUSENTE")
print("-" * 60)

display(
    uenp_alvo.loc[
        uenp_alvo["kg_leite_anterior"].isna(),
        "situacao_anterior"
    ].value_counts(dropna=False)
)

### 2.5 Harmonização da variável de produção anterior

A comparação entre as propriedades demonstrou diferença na representação da ausência de uma produção anterior válida.

Em Cerqueira César, foram identificados 82 registros com valor igual a zero na variável de produção anterior. A análise realizada durante a preparação da propriedade indicou que esses valores estão associados predominantemente ao início de uma nova lactação, sendo utilizados como convenção de registro e não como uma produção efetivamente mensurada de zero quilograma.

Na UENP, a ausência de uma produção anterior válida foi representada por valores ausentes. Esses casos correspondem principalmente ao primeiro controle disponível do animal ou a controles precedidos por registros de secagem, ausência ou falta de mensuração.

Para tornar as duas fontes semanticamente compatíveis, os valores iguais a zero de Cerqueira César foram tratados como ausência de produção anterior na variável harmonizada.

Foi também criada uma variável indicadora de disponibilidade da produção anterior. Dessa forma, o valor numérico foi utilizado somente quando existir uma mensuração anterior válida, preservando separadamente a informação de sua disponibilidade.

In [ ]:
# ============================================================
# 2.5 HARMONIZAÇÃO DA PRODUÇÃO ANTERIOR
# ============================================================

# ------------------------------------------------------------
# CERQUEIRA CÉSAR
# ------------------------------------------------------------

cc_harm["producao_anterior"] = (
    cc_harm["kg_anterior"]
    .where(cc_harm["kg_anterior"] > 0)
)

cc_harm["producao_anterior_disponivel"] = (
    cc_harm["producao_anterior"]
    .notna()
    .astype(int)
)


# ------------------------------------------------------------
# UENP
# ------------------------------------------------------------

uenp_harm["producao_anterior"] = (
    uenp_harm["kg_leite_anterior"]
)

uenp_harm["producao_anterior_disponivel"] = (
    uenp_harm["producao_anterior"]
    .notna()
    .astype(int)
)


# ------------------------------------------------------------
# SOMENTE OBSERVAÇÕES COM ALVO
# ------------------------------------------------------------

cc_alvo = cc_harm.loc[
    cc_harm["kg_leite"].notna()
].copy()

uenp_alvo = uenp_harm.loc[
    uenp_harm["kg_leite"].notna()
].copy()


# ------------------------------------------------------------
# VALIDAÇÃO
# ------------------------------------------------------------

print("PRODUÇÃO ANTERIOR HARMONIZADA")
print("-" * 60)

for nome, base in [
    ("Cerqueira César", cc_alvo),
    ("UENP", uenp_alvo)
]:

    preenchidos = (
        base["producao_anterior"]
        .notna()
        .sum()
    )

    ausentes = (
        base["producao_anterior"]
        .isna()
        .sum()
    )

    print(f"\n{nome}")
    print("-" * 30)

    print(
        "Produção anterior disponível:",
        preenchidos
    )

    print(
        "Produção anterior ausente:",
        ausentes
    )

    print(
        "Cobertura:",
        f"{preenchidos / len(base) * 100:.2f}%"
    )

    print(
        "Zeros após harmonização:",
        (base["producao_anterior"] == 0).sum()
    )


# ------------------------------------------------------------
# COBERTURA CONJUNTA
# ------------------------------------------------------------

total_preenchido = (
    cc_alvo["producao_anterior"].notna().sum()
    + uenp_alvo["producao_anterior"].notna().sum()
)

total_observacoes = (
    len(cc_alvo)
    + len(uenp_alvo)
)

print("\nCONJUNTO DAS DUAS PROPRIEDADES")
print("-" * 60)

print(
    "Disponível:",
    f"{total_preenchido}/{total_observacoes}"
)

print(
    "Cobertura:",
    f"{total_preenchido / total_observacoes * 100:.2f}%"
)

### 2.6 Construção da base harmonizada

Após a avaliação da equivalência conceitual das variáveis, as observações com produção de leite mensurada das duas propriedades foram reunidas em uma única estrutura.

As variáveis com significado equivalente foram padronizadas para nomes comuns. Os dias em lactação foram representados pela variável `del_harmonizado`, utilizando o valor original em Cerqueira César e o valor temporalmente derivado na UENP.

A produção anterior foi utilizada em sua versão harmonizada, na qual somente produções anteriores efetivamente válidas permanecem como valores numéricos.

Foi também criado um identificador global do animal, formado pela combinação entre propriedade e identificador individual. Essa estratégia evita possíveis colisões entre códigos utilizados independentemente nas duas propriedades e permitiu controlar a dependência entre observações do mesmo animal durante a validação dos modelos.

O peso corporal, disponível exclusivamente na UENP, foi preservado na base harmonizada, mas não integrou os atributos dos modelos conjuntos. As informações reprodutivas exclusivas de Cerqueira César permaneceram na base processada de origem.

Nenhuma observação foi excluída nesta etapa devido à ausência de atributos preditores.

In [ ]:
# ============================================================
# 2.6 CONSTRUÇÃO DA BASE HARMONIZADA
# ============================================================

# ------------------------------------------------------------
# 1. CERQUEIRA CÉSAR
# ------------------------------------------------------------

cc_ml_harm = cc_alvo.copy()

cc_ml_harm["id_animal_original"] = (
    cc_ml_harm["brinco"]
    .astype("string")
    .str.strip()
)

cc_ml_harm["id_animal_global"] = (
    "CC_"
    + cc_ml_harm["id_animal_original"]
)

cc_ml_harm["del_harmonizado"] = (
    cc_ml_harm["del"]
)

cc_ml_harm["raca_harmonizada"] = (
    cc_ml_harm["raca_padronizada"]
)

# Variável exclusiva da UENP
cc_ml_harm["peso_kg"] = np.nan


# ------------------------------------------------------------
# 2. UENP
# ------------------------------------------------------------

uenp_ml_harm = uenp_alvo.copy()

uenp_ml_harm["id_animal_original"] = (
    uenp_ml_harm["Número"]
    .astype("string")
    .str.strip()
)

uenp_ml_harm["id_animal_global"] = (
    "UENP_"
    + uenp_ml_harm["id_animal_original"]
)

uenp_ml_harm["del_harmonizado"] = (
    uenp_ml_harm["del_derivado"]
)

uenp_ml_harm["raca_harmonizada"] = (
    uenp_ml_harm["raca_padronizada"]
)


# ------------------------------------------------------------
# 3. SELEÇÃO DA ESTRUTURA COMUM
# ------------------------------------------------------------

colunas_harmonizadas = [
    "propriedade",
    "id_animal_original",
    "id_animal_global",
    "data_prova",

    # variável-alvo
    "kg_leite",

    # preditoras harmonizadas
    "producao_anterior",
    "producao_anterior_disponivel",
    "del_harmonizado",
    "raca_harmonizada",
    "idade_anos",
    "mes_controle",

    # variável exclusiva da UENP
    "peso_kg"
]

cc_ml_harm = (
    cc_ml_harm
    .reindex(columns=colunas_harmonizadas)
)

uenp_ml_harm = (
    uenp_ml_harm
    .reindex(columns=colunas_harmonizadas)
)


# ------------------------------------------------------------
# 4. UNIÃO DAS PROPRIEDADES
# ------------------------------------------------------------

base_harmonizada = pd.concat(
    [
        cc_ml_harm,
        uenp_ml_harm
    ],
    ignore_index=True
)

base_harmonizada = (
    base_harmonizada
    .sort_values(
        [
            "propriedade",
            "id_animal_global",
            "data_prova"
        ]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 5. VALIDAÇÃO INICIAL
# ------------------------------------------------------------

print("BASE HARMONIZADA")
print("-" * 60)

print(
    "Observações:",
    len(base_harmonizada)
)

print(
    "Colunas:",
    base_harmonizada.shape[1]
)

print(
    "Animais distintos:",
    base_harmonizada[
        "id_animal_global"
    ].nunique()
)

print(
    "Produções válidas:",
    base_harmonizada[
        "kg_leite"
    ].notna().sum()
)

print(
    "Duplicidades propriedade + animal + data:",
    base_harmonizada.duplicated(
        subset=[
            "propriedade",
            "id_animal_global",
            "data_prova"
        ]
    ).sum()
)


print("\nOBSERVAÇÕES POR PROPRIEDADE")
print("-" * 60)

display(
    base_harmonizada[
        "propriedade"
    ].value_counts()
)


print("\nCOBERTURA DAS VARIÁVEIS")
print("-" * 60)

for coluna in [
    "kg_leite",
    "producao_anterior",
    "del_harmonizado",
    "raca_harmonizada",
    "idade_anos",
    "mes_controle",
    "peso_kg"
]:

    n = (
        base_harmonizada[
            coluna
        ].notna().sum()
    )

    print(
        f"{coluna}: "
        f"{n}/{len(base_harmonizada)} "
        f"({n / len(base_harmonizada) * 100:.2f}%)"
    )

### 2.7 Avaliação da cobertura dos cenários de modelagem

Embora a base harmonizada contenha 936 observações com produção de leite mensurada, parte das variáveis preditoras apresenta valores ausentes.

A cobertura individual de cada atributo não é suficiente para determinar quantas observações poderiam ser utilizadas simultaneamente, pois os valores ausentes podem ocorrer em diferentes registros.

Por esse motivo, foram avaliadas diferentes combinações de atributos. O objetivo é quantificar a quantidade de observações e animais disponíveis em cada possível cenário antes da definição dos experimentos de Aprendizado de Máquina.

Nesta etapa nenhuma observação foi removida definitivamente. A análise serviu apenas para orientar a escolha dos conjuntos de atributos.

In [ ]:
# ============================================================
# 2.7 COBERTURA DOS CENÁRIOS DE MODELAGEM
# ============================================================

cenarios = {
    "C0_mes": [
        "mes_controle"
    ],

    "C1_historico": [
        "mes_controle",
        "producao_anterior"
    ],

    "C2_historico_DEL": [
        "mes_controle",
        "producao_anterior",
        "del_harmonizado"
    ],

    "C3_historico_DEL_idade": [
        "mes_controle",
        "producao_anterior",
        "del_harmonizado",
        "idade_anos"
    ],

    "C4_historico_DEL_raca": [
        "mes_controle",
        "producao_anterior",
        "del_harmonizado",
        "raca_harmonizada"
    ],

    "C5_completo_comum": [
        "mes_controle",
        "producao_anterior",
        "del_harmonizado",
        "idade_anos",
        "raca_harmonizada"
    ]
}


resultados_cenarios = []

for nome, variaveis in cenarios.items():

    mascara_completa = (
        base_harmonizada[variaveis]
        .notna()
        .all(axis=1)
    )

    base_cenario = base_harmonizada.loc[
        mascara_completa
    ]

    resultados_cenarios.append({
        "cenario": nome,
        "variaveis": ", ".join(variaveis),
        "observacoes": len(base_cenario),
        "percentual_base": round(
            len(base_cenario)
            / len(base_harmonizada)
            * 100,
            2
        ),
        "animais": (
            base_cenario[
                "id_animal_global"
            ].nunique()
        ),
        "CC": (
            base_cenario[
                "propriedade"
            ].eq("Cerqueira César")
            .sum()
        ),
        "UENP": (
            base_cenario[
                "propriedade"
            ].eq("UENP")
            .sum()
        )
    })


cobertura_cenarios = pd.DataFrame(
    resultados_cenarios
)

print("COBERTURA DOS CENÁRIOS")
print("-" * 60)

display(cobertura_cenarios)

In [ ]:
# ============================================================
# 2.8 PADRÕES DE AUSÊNCIA DAS VARIÁVEIS HARMONIZADAS
# ============================================================

variaveis_modelagem = [
    "producao_anterior",
    "del_harmonizado",
    "idade_anos",
    "raca_harmonizada"
]

padrao_ausencias = (
    base_harmonizada[
        variaveis_modelagem
    ]
    .isna()
    .value_counts()
    .reset_index(name="quantidade")
)

padrao_ausencias["percentual"] = (
    padrao_ausencias["quantidade"]
    / len(base_harmonizada)
    * 100
).round(2)

print("PADRÕES DE AUSÊNCIA")
print("-" * 60)

display(padrao_ausencias)

### 2.9 Definição dos cenários de modelagem

A análise de cobertura demonstrou que a utilização exclusiva de registros completos resultaria em perda considerável de informações.

O cenário contendo todas as variáveis comuns completas preservaria apenas 479 das 936 observações disponíveis, correspondendo a 51,18% da base harmonizada. Além disso, apenas 68 observações da UENP permaneceriam nesse subconjunto, ocasionando maior predominância dos registros provenientes de Cerqueira César.

Por esse motivo, a exclusão de registros incompletos não foi utilizada como estratégia principal de modelagem.

Os valores ausentes foram tratados dentro dos pipelines de Aprendizado de Máquina, garantindo que os parâmetros necessários à imputação fossem estimados exclusivamente a partir dos dados de treinamento.

Também foi preservada a informação sobre a disponibilidade de atributos cuja ausência possui significado ou decorre das limitações das fontes.

A propriedade de origem foi incluída como atributo categórico nos modelos conjuntos, permitindo representar possíveis diferenças entre os sistemas produtivos.

Foram avaliados progressivamente diferentes conjuntos de atributos, iniciando por um modelo de referência e acrescentando informações produtivas, de lactação e características individuais.

In [ ]:
# ============================================================
# 2.9 INDICADORES DE DISPONIBILIDADE E CENÁRIOS DEFINITIVOS
# ============================================================

# Produção anterior já possui indicador criado anteriormente.
# Conferimos apenas se está consistente.
base_harmonizada["producao_anterior_disponivel"] = (
    base_harmonizada["producao_anterior"]
    .notna()
    .astype(int)
)


# ------------------------------------------------------------
# INDICADOR DE DEL
# ------------------------------------------------------------

base_harmonizada["del_disponivel"] = (
    base_harmonizada["del_harmonizado"]
    .notna()
    .astype(int)
)


# ------------------------------------------------------------
# INDICADOR DE IDADE
# ------------------------------------------------------------

base_harmonizada["idade_disponivel"] = (
    base_harmonizada["idade_anos"]
    .notna()
    .astype(int)
)


# ------------------------------------------------------------
# INDICADOR DE RAÇA
# ------------------------------------------------------------

base_harmonizada["raca_disponivel"] = (
    base_harmonizada["raca_harmonizada"]
    .notna()
    .astype(int)
)


# ------------------------------------------------------------
# CENÁRIOS DEFINITIVOS
# ------------------------------------------------------------

cenarios_ml = {

    "C0_baseline": [
        "propriedade",
        "mes_controle"
    ],

    "C1_historico": [
        "propriedade",
        "mes_controle",
        "producao_anterior",
        "producao_anterior_disponivel"
    ],

    "C2_lactacao": [
        "propriedade",
        "mes_controle",
        "producao_anterior",
        "producao_anterior_disponivel",
        "del_harmonizado",
        "del_disponivel"
    ],

    "C3_completo_comum": [
        "propriedade",
        "mes_controle",
        "producao_anterior",
        "producao_anterior_disponivel",
        "del_harmonizado",
        "del_disponivel",
        "idade_anos",
        "idade_disponivel",
        "raca_harmonizada",
        "raca_disponivel"
    ]
}


print("CENÁRIOS DEFINITIVOS DE MODELAGEM")
print("-" * 60)

for nome, variaveis in cenarios_ml.items():

    print(f"\n{nome}")

    for variavel in variaveis:
        print(" -", variavel)


# ------------------------------------------------------------
# VALIDAÇÃO DOS INDICADORES
# ------------------------------------------------------------

print("\nCOBERTURA / DISPONIBILIDADE")
print("-" * 60)

for variavel, indicador in [
    ("producao_anterior", "producao_anterior_disponivel"),
    ("del_harmonizado", "del_disponivel"),
    ("idade_anos", "idade_disponivel"),
    ("raca_harmonizada", "raca_disponivel")
]:

    preenchidos = base_harmonizada[variavel].notna().sum()
    marcados = base_harmonizada[indicador].sum()

    print(
        f"{variavel}: "
        f"{preenchidos} preenchidos | "
        f"{marcados} indicados como disponíveis | "
        f"consistente = {preenchidos == marcados}"
    )

# 3. Caracterização da variável-alvo e estrutura temporal

Antes da divisão dos dados em conjuntos de treinamento e teste, foi caracterizada a distribuição da produção de leite e a estrutura temporal da base harmonizada.

Essa análise é necessária porque as observações não são independentes: um mesmo animal pode possuir diversos controles leiteiros ao longo do tempo.

Além disso, as duas propriedades apresentam diferentes períodos e quantidades de registros. Essas características devem ser consideradas na estratégia de validação para evitar que a avaliação dos modelos seja artificialmente otimista.

Nesta etapa foram avaliados a distribuição da produção de leite, o período coberto por cada propriedade, a quantidade de animais e o número de observações disponíveis por animal.

In [ ]:
# ============================================================
# 3. CARACTERIZAÇÃO DO ALVO E DA ESTRUTURA TEMPORAL
# ============================================================

print("CARACTERIZAÇÃO DA BASE HARMONIZADA")
print("-" * 60)


# ------------------------------------------------------------
# 1. PRODUÇÃO DE LEITE POR PROPRIEDADE
# ------------------------------------------------------------

print("\nPRODUÇÃO DE LEITE POR PROPRIEDADE")
print("-" * 60)

resumo_producao = (
    base_harmonizada
    .groupby("propriedade")["kg_leite"]
    .agg(
        observacoes="count",
        media="mean",
        desvio_padrao="std",
        minimo="min",
        mediana="median",
        maximo="max"
    )
    .round(2)
)

display(resumo_producao)


# ------------------------------------------------------------
# 2. PERÍODO DISPONÍVEL
# ------------------------------------------------------------

print("\nPERÍODO DOS CONTROLES")
print("-" * 60)

periodo = (
    base_harmonizada
    .groupby("propriedade")
    .agg(
        data_inicial=("data_prova", "min"),
        data_final=("data_prova", "max"),
        observacoes=("kg_leite", "size"),
        animais=("id_animal_global", "nunique")
    )
)

display(periodo)


# ------------------------------------------------------------
# 3. OBSERVAÇÕES POR ANO E PROPRIEDADE
# ------------------------------------------------------------

base_harmonizada["ano_controle"] = (
    base_harmonizada["data_prova"].dt.year
)

print("\nOBSERVAÇÕES POR ANO")
print("-" * 60)

display(
    pd.crosstab(
        base_harmonizada["ano_controle"],
        base_harmonizada["propriedade"],
        margins=True
    )
)


# ------------------------------------------------------------
# 4. NÚMERO DE CONTROLES POR ANIMAL
# ------------------------------------------------------------

controles_por_animal = (
    base_harmonizada
    .groupby(
        [
            "propriedade",
            "id_animal_global"
        ]
    )
    .size()
    .reset_index(name="numero_controles")
)

print("\nCONTROLES POR ANIMAL")
print("-" * 60)

display(
    controles_por_animal
    .groupby("propriedade")["numero_controles"]
    .describe()
    .round(2)
)


# ------------------------------------------------------------
# 5. ANIMAIS COM APENAS UM CONTROLE
# ------------------------------------------------------------

print("\nANIMAIS COM APENAS UM CONTROLE")
print("-" * 60)

um_controle = (
    controles_por_animal[
        "numero_controles"
    ] == 1
)

print(
    "Total:",
    um_controle.sum()
)

display(
    controles_por_animal.loc[
        um_controle,
        "propriedade"
    ].value_counts()
)


# ------------------------------------------------------------
# 6. DISTRIBUIÇÃO GLOBAL DO ALVO
# ------------------------------------------------------------

print("\nPRODUÇÃO DE LEITE — BASE COMPLETA")
print("-" * 60)

display(
    base_harmonizada[
        "kg_leite"
    ].describe()
)

## 3.1 Estratégia de separação entre treinamento e teste

As observações da base apresentam estrutura longitudinal, uma vez que um mesmo animal pode possuir diversos controles leiteiros.

Uma separação aleatória convencional entre linhas poderia distribuir controles de um mesmo animal simultaneamente entre os conjuntos de treinamento e teste. Essa situação poderia produzir uma avaliação excessivamente otimista, pois as observações de um mesmo indivíduo tendem a apresentar características relacionadas.

Por esse motivo, a avaliação principal foi realizada por animal. Todos os controles de um mesmo indivíduo foram mantidos exclusivamente no conjunto de treinamento ou exclusivamente no conjunto de teste.

A separação foi realizada independentemente dentro de cada propriedade, preservando a presença de animais das duas fontes em ambos os conjuntos.



In [ ]:
# ============================================================
# 3.1 SEPARAÇÃO TREINO/TESTE POR ANIMAL
# ============================================================

from sklearn.model_selection import train_test_split

SEMENTE = 42


# ------------------------------------------------------------
# 1. LISTA ÚNICA DE ANIMAIS POR PROPRIEDADE
# ------------------------------------------------------------

animais_por_propriedade = (
    base_harmonizada[
        [
            "id_animal_global",
            "propriedade"
        ]
    ]
    .drop_duplicates()
)


# ------------------------------------------------------------
# 2. SEPARAÇÃO DOS ANIMAIS DE CADA PROPRIEDADE
# ------------------------------------------------------------

animais_treino = []
animais_teste = []

for propriedade, grupo in animais_por_propriedade.groupby(
    "propriedade"
):

    ids = grupo[
        "id_animal_global"
    ].tolist()

    ids_treino, ids_teste = train_test_split(
        ids,
        test_size=0.20,
        random_state=SEMENTE
    )

    animais_treino.extend(ids_treino)
    animais_teste.extend(ids_teste)


# ------------------------------------------------------------
# 3. CRIAÇÃO DAS BASES
# ------------------------------------------------------------

base_treino = base_harmonizada.loc[
    base_harmonizada[
        "id_animal_global"
    ].isin(animais_treino)
].copy()

base_teste = base_harmonizada.loc[
    base_harmonizada[
        "id_animal_global"
    ].isin(animais_teste)
].copy()


# ------------------------------------------------------------
# 4. VALIDAÇÃO DE VAZAMENTO ENTRE ANIMAIS
# ------------------------------------------------------------

animais_em_comum = (
    set(base_treino["id_animal_global"])
    &
    set(base_teste["id_animal_global"])
)


print("SEPARAÇÃO TREINO/TESTE POR ANIMAL")
print("-" * 60)

print(
    "Observações de treino:",
    len(base_treino)
)

print(
    "Observações de teste:",
    len(base_teste)
)

print(
    "Animais de treino:",
    base_treino[
        "id_animal_global"
    ].nunique()
)

print(
    "Animais de teste:",
    base_teste[
        "id_animal_global"
    ].nunique()
)

print(
    "Animais presentes nos dois conjuntos:",
    len(animais_em_comum)
)


# ------------------------------------------------------------
# 5. DISTRIBUIÇÃO POR PROPRIEDADE
# ------------------------------------------------------------

print("\nOBSERVAÇÕES POR PROPRIEDADE")
print("-" * 60)

display(
    pd.DataFrame({
        "treino": (
            base_treino[
                "propriedade"
            ].value_counts()
        ),
        "teste": (
            base_teste[
                "propriedade"
            ].value_counts()
        )
    }).fillna(0).astype(int)
)


print("\nANIMAIS POR PROPRIEDADE")
print("-" * 60)

display(
    pd.DataFrame({
        "treino": (
            base_treino
            .groupby("propriedade")[
                "id_animal_global"
            ]
            .nunique()
        ),
        "teste": (
            base_teste
            .groupby("propriedade")[
                "id_animal_global"
            ]
            .nunique()
        )
    }).fillna(0).astype(int)
)


# ------------------------------------------------------------
# 6. DISTRIBUIÇÃO DO ALVO
# ------------------------------------------------------------

print("\nPRODUÇÃO DE LEITE")
print("-" * 60)

resumo_split = pd.DataFrame({
    "treino": base_treino["kg_leite"].describe(),
    "teste": base_teste["kg_leite"].describe()
}).round(2)

display(resumo_split)

## 3.2 Validação cruzada agrupada por animal

Após a separação do conjunto de teste, o desenvolvimento e a comparação dos modelos foram realizados exclusivamente utilizando o conjunto de treinamento.

Para evitar que diferentes controles de um mesmo animal sejam distribuídos entre treinamento e validação, foi empregada validação cruzada agrupada pelo identificador individual.

Também foi considerada a propriedade de origem na formação dos folds, buscando preservar a representação das duas propriedades ao longo das divisões.

Foram utilizados cinco folds. Em cada divisão, todos os registros de um determinado animal permaneceram exclusivamente no subconjunto de treinamento ou de validação.

O conjunto de teste previamente separado não participou desta etapa e foi utilizado somente após a seleção dos modelos.

In [ ]:
# ============================================================
# 3.2 VALIDAÇÃO CRUZADA AGRUPADA POR ANIMAL
# ============================================================

from sklearn.model_selection import StratifiedGroupKFold

N_FOLDS = 5

cv_agrupada = StratifiedGroupKFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=SEMENTE
)

grupos_treino = base_treino[
    "id_animal_global"
]

# A propriedade será usada apenas para ajudar a distribuir
# as duas fazendas entre os folds.
estratos_propriedade = base_treino[
    "propriedade"
]


# ------------------------------------------------------------
# INSPEÇÃO DOS FOLDS
# ------------------------------------------------------------

resultados_folds = []

vazamento_encontrado = False

for fold, (idx_treino, idx_validacao) in enumerate(
    cv_agrupada.split(
        base_treino,
        estratos_propriedade,
        groups=grupos_treino
    ),
    start=1
):

    treino_fold = base_treino.iloc[
        idx_treino
    ]

    validacao_fold = base_treino.iloc[
        idx_validacao
    ]

    animais_fold_treino = set(
        treino_fold["id_animal_global"]
    )

    animais_fold_validacao = set(
        validacao_fold["id_animal_global"]
    )

    animais_comuns = (
        animais_fold_treino
        & animais_fold_validacao
    )

    if len(animais_comuns) > 0:
        vazamento_encontrado = True

    resultados_folds.append({
        "fold": fold,

        "obs_treino": len(treino_fold),
        "obs_validacao": len(validacao_fold),

        "animais_treino": len(
            animais_fold_treino
        ),

        "animais_validacao": len(
            animais_fold_validacao
        ),

        "animais_em_comum": len(
            animais_comuns
        ),

        "CC_validacao": (
            validacao_fold[
                "propriedade"
            ] == "Cerqueira César"
        ).sum(),

        "UENP_validacao": (
            validacao_fold[
                "propriedade"
            ] == "UENP"
        ).sum(),

        "media_kg_validacao": round(
            validacao_fold[
                "kg_leite"
            ].mean(),
            2
        )
    })


resumo_folds = pd.DataFrame(
    resultados_folds
)

print("VALIDAÇÃO CRUZADA AGRUPADA")
print("-" * 60)

display(resumo_folds)


print("\nVALIDAÇÃO")
print("-" * 60)

print(
    "Número de folds:",
    len(resumo_folds)
)

print(
    "Algum animal apareceu simultaneamente "
    "no treino e validação?:",
    vazamento_encontrado
)

print(
    "Todos os folds possuem Cerqueira César?:",
    (resumo_folds["CC_validacao"] > 0).all()
)

print(
    "Todos os folds possuem UENP?:",
    (resumo_folds["UENP_validacao"] > 0).all()
)

# 4. Treinamento e avaliação dos modelos

Antes da avaliação dos algoritmos de Aprendizado de Máquina, foi estabelecido um modelo de referência (baseline).

O baseline não utiliza os atributos preditores para identificar relações entre as características dos animais e a produção. Em cada divisão da validação cruzada, ele estima a produção utilizando apenas a média da variável-alvo observada no subconjunto de treinamento.

Seu desempenho funciona como referência mínima para a comparação dos modelos posteriores. Um modelo preditivo somente apresenta ganho prático se produzir erros menores do que essa estratégia simples.

Foram utilizadas como principais métricas:

- **MAE (Mean Absolute Error):** representa, em quilogramas de leite, o erro absoluto médio das previsões;
- **RMSE (Root Mean Squared Error):** também expresso em quilogramas, penalizando mais intensamente erros de maior magnitude;
- **R² (coeficiente de determinação):** indica a proporção da variação da produção explicada pelo modelo.

O baseline foi avaliado utilizando os mesmos cinco folds agrupados por animal definidos anteriormente.

In [ ]:
# ============================================================
# 4.1 BASELINE — PREVISÃO PELA MÉDIA
# ============================================================

from sklearn.dummy import DummyRegressor
from sklearn.model_selection import cross_validate


# ------------------------------------------------------------
# 1. DADOS
# ------------------------------------------------------------

y_treino = base_treino["kg_leite"]

# O DummyRegressor não utiliza atributos de verdade.
# Criamos apenas uma coluna fictícia.
X_baseline = np.zeros(
    (len(base_treino), 1)
)


# ------------------------------------------------------------
# 2. PRÉ-CÁLCULO DOS FOLDS
# ------------------------------------------------------------
#
# IMPORTANTE:
# StratifiedGroupKFold precisa de uma variável categórica
# para realizar a estratificação.
#
# Aqui usamos a propriedade apenas para formar os folds.
# kg_leite continua sendo a variável-alvo da regressão.
# ------------------------------------------------------------

splits_cv = list(
    cv_agrupada.split(
        base_treino,
        estratos_propriedade,
        groups=grupos_treino
    )
)

print(
    "Folds preparados:",
    len(splits_cv)
)


# ------------------------------------------------------------
# 3. MODELO BASELINE
# ------------------------------------------------------------

modelo_baseline = DummyRegressor(
    strategy="mean"
)


# ------------------------------------------------------------
# 4. VALIDAÇÃO CRUZADA
# ------------------------------------------------------------

resultado_baseline_cv = cross_validate(
    modelo_baseline,
    X_baseline,
    y_treino,

    cv=splits_cv,

    scoring={
        "MAE": "neg_mean_absolute_error",
        "RMSE": "neg_root_mean_squared_error",
        "R2": "r2"
    },

    return_train_score=False
)


# ------------------------------------------------------------
# 5. CONVERSÃO DAS MÉTRICAS
# ------------------------------------------------------------

mae_baseline = (
    -resultado_baseline_cv[
        "test_MAE"
    ]
)

rmse_baseline = (
    -resultado_baseline_cv[
        "test_RMSE"
    ]
)

r2_baseline = (
    resultado_baseline_cv[
        "test_R2"
    ]
)


# ------------------------------------------------------------
# 6. RESULTADO POR FOLD
# ------------------------------------------------------------

baseline_folds = pd.DataFrame({
    "fold": range(1, N_FOLDS + 1),
    "MAE": mae_baseline,
    "RMSE": rmse_baseline,
    "R2": r2_baseline
}).round(3)

print("\nBASELINE — RESULTADO POR FOLD")
print("-" * 60)

display(baseline_folds)


# ------------------------------------------------------------
# 7. RESULTADO MÉDIO
# ------------------------------------------------------------

resumo_baseline = pd.DataFrame({
    "modelo": [
        "Baseline — média"
    ],

    "MAE_medio": [
        mae_baseline.mean()
    ],

    "MAE_desvio": [
        mae_baseline.std()
    ],

    "RMSE_medio": [
        rmse_baseline.mean()
    ],

    "RMSE_desvio": [
        rmse_baseline.std()
    ],

    "R2_medio": [
        r2_baseline.mean()
    ],

    "R2_desvio": [
        r2_baseline.std()
    ]
}).round(3)


print("\nBASELINE — RESUMO")
print("-" * 60)

display(resumo_baseline)

## 4.2 Regressão Linear

A Regressão Linear foi utilizada inicialmente como modelo preditivo de referência, permitindo avaliar o ganho obtido pela incorporação progressiva dos diferentes grupos de atributos.

Foram avaliados os quatro cenários previamente definidos. Dessa forma, as diferenças de desempenho observadas entre os cenários podem ser atribuídas principalmente à disponibilidade de novas informações, uma vez que o algoritmo utilizado permanece o mesmo.

O pré-processamento foi realizado dentro de um `Pipeline`, garantindo que todas as transformações sejam ajustadas exclusivamente sobre os dados de treinamento de cada fold.

Para atributos numéricos ausentes foi utilizada imputação pela mediana do subconjunto de treinamento. As variáveis categóricas foram preenchidas pela categoria mais frequente e transformadas por codificação one-hot.

Os dados numéricos também foram padronizados para manter uma estrutura de pré-processamento adequada aos modelos lineares.

A avaliação foi realizada utilizando os mesmos cinco folds agrupados por animal definidos anteriormente. O conjunto de teste permaneceu reservado durante esta etapa.

In [ ]:
# ============================================================
# 4.2 REGRESSÃO LINEAR — COMPARAÇÃO ENTRE CENÁRIOS
# ============================================================

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_validate


# ------------------------------------------------------------
# 1. CLASSIFICAÇÃO DOS TIPOS DE VARIÁVEL
# ------------------------------------------------------------

variaveis_categoricas = {
    "propriedade",
    "raca_harmonizada"
}

variaveis_numericas = {
    "mes_controle",
    "producao_anterior",
    "producao_anterior_disponivel",
    "del_harmonizado",
    "del_disponivel",
    "idade_anos",
    "idade_disponivel",
    "raca_disponivel"
}


# ------------------------------------------------------------
# 2. FUNÇÃO PARA CRIAR O PIPELINE
# ------------------------------------------------------------

def criar_pipeline_regressao_linear(variaveis):

    numericas = [
        v for v in variaveis
        if v in variaveis_numericas
    ]

    categoricas = [
        v for v in variaveis
        if v in variaveis_categoricas
    ]


    # ---------------- NUMÉRICAS ----------------

    pipeline_numerico = Pipeline([
        (
            "imputacao",
            SimpleImputer(
                strategy="median"
            )
        ),
        (
            "padronizacao",
            StandardScaler()
        )
    ])


    # ---------------- CATEGÓRICAS ----------------

    pipeline_categorico = Pipeline([
        (
            "imputacao",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ])


    # ---------------- PRÉ-PROCESSAMENTO ----------------

    preprocessador = ColumnTransformer([
        (
            "numericas",
            pipeline_numerico,
            numericas
        ),
        (
            "categoricas",
            pipeline_categorico,
            categoricas
        )
    ])


    # ---------------- MODELO ----------------

    pipeline = Pipeline([
        (
            "preprocessamento",
            preprocessador
        ),
        (
            "modelo",
            LinearRegression()
        )
    ])

    return pipeline


# ------------------------------------------------------------
# 3. AVALIAÇÃO DOS QUATRO CENÁRIOS
# ------------------------------------------------------------

resultados_regressao_linear = []

resultados_regressao_linear_folds = {}

y_treino = base_treino["kg_leite"]


for nome_cenario, variaveis in cenarios_ml.items():

    print(
        f"Avaliando {nome_cenario}..."
    )

    X_cenario = base_treino[
        variaveis
    ].copy()

    pipeline = criar_pipeline_regressao_linear(
        variaveis
    )

    resultado = cross_validate(
        pipeline,
        X_cenario,
        y_treino,

        cv=splits_cv,

        scoring={
            "MAE": "neg_mean_absolute_error",
            "RMSE": "neg_root_mean_squared_error",
            "R2": "r2"
        },

        return_train_score=False
    )


    mae = -resultado["test_MAE"]
    rmse = -resultado["test_RMSE"]
    r2 = resultado["test_R2"]


    resultados_regressao_linear_folds[
        nome_cenario
    ] = pd.DataFrame({
        "fold": range(
            1,
            N_FOLDS + 1
        ),
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })


    resultados_regressao_linear.append({
        "modelo": "Regressão Linear",
        "cenario": nome_cenario,

        "MAE_medio": mae.mean(),
        "MAE_desvio": mae.std(),

        "RMSE_medio": rmse.mean(),
        "RMSE_desvio": rmse.std(),

        "R2_medio": r2.mean(),
        "R2_desvio": r2.std()
    })


# ------------------------------------------------------------
# 4. TABELA FINAL
# ------------------------------------------------------------

resumo_regressao_linear = (
    pd.DataFrame(
        resultados_regressao_linear
    )
    .round(3)
)

print(
    "\nREGRESSÃO LINEAR — RESULTADOS"
)

print("-" * 60)

display(
    resumo_regressao_linear
)

### 4.3 Resultados iniciais da Regressão Linear

A Regressão Linear apresentou desempenho superior ao baseline em todos os cenários avaliados.

O modelo de referência baseado exclusivamente na média da produção apresentou MAE médio de 9,274 kg, RMSE de 11,028 kg e R² médio de -0,009.

No cenário C0, contendo apenas propriedade e mês do controle, o MAE foi reduzido para 6,482 kg e o R² atingiu 0,423. Esse resultado indica que diferenças entre as propriedades e a componente temporal mensal já apresentam capacidade explicativa relevante.

A inclusão da produção anterior no cenário C1 proporcionou o maior ganho incremental observado, reduzindo o MAE para 4,368 kg e elevando o R² médio para 0,710.

No cenário C2, a inclusão dos dias em lactação resultou em nova melhora, atingindo MAE médio de 4,194 kg, RMSE de 5,555 kg e R² de 0,743. Entre os cenários avaliados por Regressão Linear, esse foi o melhor desempenho.

A adição de idade e raça no cenário C3 não produziu melhora adicional, resultando em MAE de 4,245 kg e R² de 0,741.

Esses resultados sugerem que o histórico produtivo recente e o estágio da lactação apresentam contribuição preditiva relevante para a estimativa da produção de leite, enquanto idade e raça não proporcionaram ganho adicional para o modelo linear nesta base de dados.

Esses resultados correspondem à validação cruzada sobre o conjunto de treinamento. Durante esta etapa, o conjunto de teste permaneceu reservado para a avaliação final do modelo selecionado.

### 4.4 Random Forest

Após a Regressão Linear, foi avaliado o algoritmo Random Forest Regressor.

Diferentemente do modelo linear, Random Forest é capaz de representar relações não lineares e interações entre os atributos sem exigir que essas relações sejam especificadas previamente.

Para manter a comparabilidade dos experimentos, foram utilizados os mesmos quatro cenários de atributos e os mesmos cinco folds agrupados por animal empregados na Regressão Linear.

O pré-processamento continuou sendo realizado dentro do pipeline. Valores numéricos ausentes foram imputados utilizando a mediana calculada exclusivamente no subconjunto de treinamento, enquanto atributos categóricos foram tratados por imputação e codificação one-hot.

Nesta primeira comparação foram utilizados parâmetros fixos, sem otimização específica. O ajuste de hiperparâmetros foi realizado posteriormente apenas para os modelos mais promissores.

In [ ]:
# ============================================================
# 4.4 RANDOM FOREST — COMPARAÇÃO ENTRE CENÁRIOS
# ============================================================

from sklearn.ensemble import RandomForestRegressor


# ------------------------------------------------------------
# 1. FUNÇÃO PARA CRIAR O PIPELINE
# ------------------------------------------------------------

def criar_pipeline_random_forest(variaveis):

    numericas = [
        v for v in variaveis
        if v in variaveis_numericas
    ]

    categoricas = [
        v for v in variaveis
        if v in variaveis_categoricas
    ]


    # ---------------- NUMÉRICAS ----------------

    # Árvores não precisam de padronização.
    pipeline_numerico = Pipeline([
        (
            "imputacao",
            SimpleImputer(
                strategy="median"
            )
        )
    ])


    # ---------------- CATEGÓRICAS ----------------

    pipeline_categorico = Pipeline([
        (
            "imputacao",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ])


    # ---------------- PRÉ-PROCESSAMENTO ----------------

    preprocessador = ColumnTransformer([
        (
            "numericas",
            pipeline_numerico,
            numericas
        ),
        (
            "categoricas",
            pipeline_categorico,
            categoricas
        )
    ])


    # ---------------- MODELO ----------------

    modelo = RandomForestRegressor(
        n_estimators=300,
        random_state=SEMENTE,
        n_jobs=-1
    )


    pipeline = Pipeline([
        (
            "preprocessamento",
            preprocessador
        ),
        (
            "modelo",
            modelo
        )
    ])

    return pipeline


# ------------------------------------------------------------
# 2. AVALIAÇÃO DOS CENÁRIOS
# ------------------------------------------------------------

resultados_random_forest = []
resultados_random_forest_folds = {}


for nome_cenario, variaveis in cenarios_ml.items():

    print(
        f"Avaliando {nome_cenario}..."
    )

    X_cenario = base_treino[
        variaveis
    ].copy()

    pipeline = criar_pipeline_random_forest(
        variaveis
    )

    resultado = cross_validate(
        pipeline,
        X_cenario,
        y_treino,

        cv=splits_cv,

        scoring={
            "MAE": "neg_mean_absolute_error",
            "RMSE": "neg_root_mean_squared_error",
            "R2": "r2"
        },

        return_train_score=False
    )


    mae = -resultado["test_MAE"]
    rmse = -resultado["test_RMSE"]
    r2 = resultado["test_R2"]


    resultados_random_forest_folds[
        nome_cenario
    ] = pd.DataFrame({
        "fold": range(
            1,
            N_FOLDS + 1
        ),
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })


    resultados_random_forest.append({
        "modelo": "Random Forest",
        "cenario": nome_cenario,

        "MAE_medio": mae.mean(),
        "MAE_desvio": mae.std(),

        "RMSE_medio": rmse.mean(),
        "RMSE_desvio": rmse.std(),

        "R2_medio": r2.mean(),
        "R2_desvio": r2.std()
    })


# ------------------------------------------------------------
# 3. RESULTADOS
# ------------------------------------------------------------

resumo_random_forest = (
    pd.DataFrame(
        resultados_random_forest
    )
    .round(3)
)

print(
    "\nRANDOM FOREST — RESULTADOS"
)

print("-" * 60)

display(
    resumo_random_forest
)


# ------------------------------------------------------------
# 4. COMPARAÇÃO COM REGRESSÃO LINEAR
# ------------------------------------------------------------

comparacao_modelos = pd.concat(
    [
        resumo_regressao_linear,
        resumo_random_forest
    ],
    ignore_index=True
)

comparacao_modelos = (
    comparacao_modelos
    .sort_values("MAE_medio")
    .reset_index(drop=True)
)

print(
    "\nCOMPARAÇÃO ATUAL DOS MODELOS"
)

print("-" * 60)

display(
    comparacao_modelos
)

### 4.5 Resultados iniciais do Random Forest

O Random Forest apresentou desempenho semelhante ao da Regressão Linear, com diferenças entre os cenários avaliados.

O melhor desempenho em MAE foi obtido no cenário C3, composto pelas informações de propriedade, mês do controle, produção anterior, dias em lactação, idade e raça. Nesse cenário, o modelo apresentou MAE médio de 4,071 kg, RMSE de 5,550 kg e R² de 0,742.

Em comparação ao baseline baseado na média da produção, cujo MAE foi de 9,274 kg, o Random Forest no cenário C3 reduziu o erro absoluto médio em aproximadamente 56%.

Entretanto, os resultados dos melhores modelos permaneceram próximos. A Regressão Linear no cenário C2 apresentou MAE de 4,194 kg, RMSE de 5,555 kg e R² de 0,743. Dessa forma, essa comparação inicial não permitiu considerar a pequena diferença entre os modelos como evidência suficiente de superioridade de um algoritmo.

Também foi observada diferença no efeito da inclusão de idade e raça. Na Regressão Linear, a passagem do cenário C2 para C3 não melhorou o desempenho. No Random Forest, por outro lado, o cenário C3 apresentou desempenho ligeiramente superior ao C2. Esse resultado sugere a possibilidade de relações não lineares ou interações envolvendo essas características.

A seleção do modelo final foi realizada após a comparação com os demais algoritmos e o ajuste de hiperparâmetros dos modelos mais promissores, exclusivamente com base na validação cruzada sobre o conjunto de treinamento. A avaliação no conjunto de teste reservado ocorreu após essa seleção.

### 4.6 Gradient Boosting

Como terceiro algoritmo preditivo foi avaliado o Gradient Boosting Regressor.

O método utiliza uma sequência de árvores de decisão, em que cada nova árvore é ajustada com o objetivo de reduzir os erros produzidos pelo conjunto anterior. Essa característica permite representar relações não lineares e interações entre atributos.

Os mesmos quatro cenários de atributos e os mesmos cinco folds agrupados por animal foram utilizados, mantendo a comparabilidade com a Regressão Linear e o Random Forest.

Nesta etapa foram utilizados parâmetros iniciais fixos, sem otimização específica de hiperparâmetros. O ajuste foi realizado posteriormente apenas para os algoritmos que apresentaram melhor desempenho na validação cruzada.

In [ ]:
# ============================================================
# 4.6 GRADIENT BOOSTING — COMPARAÇÃO ENTRE CENÁRIOS
# ============================================================

from sklearn.ensemble import GradientBoostingRegressor


# ------------------------------------------------------------
# 1. FUNÇÃO PARA CRIAR O PIPELINE
# ------------------------------------------------------------

def criar_pipeline_gradient_boosting(variaveis):

    numericas = [
        v for v in variaveis
        if v in variaveis_numericas
    ]

    categoricas = [
        v for v in variaveis
        if v in variaveis_categoricas
    ]


    # ---------------- NUMÉRICAS ----------------

    pipeline_numerico = Pipeline([
        (
            "imputacao",
            SimpleImputer(
                strategy="median"
            )
        )
    ])


    # ---------------- CATEGÓRICAS ----------------

    pipeline_categorico = Pipeline([
        (
            "imputacao",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ])


    # ---------------- PRÉ-PROCESSAMENTO ----------------

    preprocessador = ColumnTransformer([
        (
            "numericas",
            pipeline_numerico,
            numericas
        ),
        (
            "categoricas",
            pipeline_categorico,
            categoricas
        )
    ])


    # ---------------- MODELO ----------------

    modelo = GradientBoostingRegressor(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        random_state=SEMENTE
    )


    pipeline = Pipeline([
        (
            "preprocessamento",
            preprocessador
        ),
        (
            "modelo",
            modelo
        )
    ])

    return pipeline


# ------------------------------------------------------------
# 2. AVALIAÇÃO DOS CENÁRIOS
# ------------------------------------------------------------

resultados_gradient_boosting = []
resultados_gradient_boosting_folds = {}


for nome_cenario, variaveis in cenarios_ml.items():

    print(
        f"Avaliando {nome_cenario}..."
    )

    X_cenario = base_treino[
        variaveis
    ].copy()

    pipeline = criar_pipeline_gradient_boosting(
        variaveis
    )

    resultado = cross_validate(
        pipeline,
        X_cenario,
        y_treino,

        cv=splits_cv,

        scoring={
            "MAE": "neg_mean_absolute_error",
            "RMSE": "neg_root_mean_squared_error",
            "R2": "r2"
        },

        return_train_score=False
    )


    mae = -resultado["test_MAE"]
    rmse = -resultado["test_RMSE"]
    r2 = resultado["test_R2"]


    resultados_gradient_boosting_folds[
        nome_cenario
    ] = pd.DataFrame({
        "fold": range(
            1,
            N_FOLDS + 1
        ),
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })


    resultados_gradient_boosting.append({
        "modelo": "Gradient Boosting",
        "cenario": nome_cenario,

        "MAE_medio": mae.mean(),
        "MAE_desvio": mae.std(),

        "RMSE_medio": rmse.mean(),
        "RMSE_desvio": rmse.std(),

        "R2_medio": r2.mean(),
        "R2_desvio": r2.std()
    })


# ------------------------------------------------------------
# 3. RESULTADOS
# ------------------------------------------------------------

resumo_gradient_boosting = (
    pd.DataFrame(
        resultados_gradient_boosting
    )
    .round(3)
)

print(
    "\nGRADIENT BOOSTING — RESULTADOS"
)

print("-" * 60)

display(
    resumo_gradient_boosting
)


# ------------------------------------------------------------
# 4. COMPARAÇÃO GERAL
# ------------------------------------------------------------

comparacao_modelos = pd.concat(
    [
        resumo_regressao_linear,
        resumo_random_forest,
        resumo_gradient_boosting
    ],
    ignore_index=True
)

comparacao_modelos = (
    comparacao_modelos
    .sort_values("MAE_medio")
    .reset_index(drop=True)
)

print(
    "\nCOMPARAÇÃO GERAL DOS MODELOS"
)

print("-" * 60)

display(
    comparacao_modelos
)

### 4.7 Comparação inicial dos algoritmos

Entre os três algoritmos avaliados, o Gradient Boosting apresentou o melhor desempenho médio na validação cruzada agrupada por animal.

O melhor resultado foi obtido no cenário C2, composto pela propriedade de origem, mês do controle, produção anterior e dias em lactação, juntamente com seus respectivos indicadores de disponibilidade.

Nesse cenário, o Gradient Boosting apresentou MAE médio de 3,799 kg, RMSE de 5,299 kg e R² médio de 0,766.

Em comparação ao baseline baseado na média da produção, cujo MAE foi de 9,274 kg, houve redução aproximada de 59% no erro absoluto médio.

A inclusão de idade e raça no cenário C3 não melhorou o Gradient Boosting. O cenário C3 apresentou MAE de 3,903 kg, ligeiramente superior ao observado no C2. Dessa forma, o conjunto mais complexo de atributos não resultou em maior capacidade preditiva.

Os resultados também demonstraram que o desempenho depende da combinação entre algoritmo e conjunto de atributos. O Random Forest apresentou seu melhor desempenho no cenário C3, enquanto a Regressão Linear e o Gradient Boosting obtiveram melhores resultados no cenário C2.

Com base nessa comparação inicial, o Gradient Boosting no cenário C2 foi considerado o principal candidato para ajuste de hiperparâmetros. O Random Forest foi mantido como modelo não linear alternativo, enquanto a Regressão Linear permaneceu como referência de modelo de menor complexidade.

Até esta etapa, o conjunto de teste reservado não havia sido utilizado.

# 5. Ajuste dos modelos mais promissores

Após a comparação inicial dos algoritmos, os modelos com melhor desempenho foram submetidos ao ajuste de hiperparâmetros.

O Gradient Boosting no cenário C2 foi selecionado como principal candidato por apresentar o menor MAE médio entre todas as configurações avaliadas.

A otimização foi realizada exclusivamente com o conjunto de treinamento e utilizando os mesmos folds agrupados por animal empregados anteriormente.

O conjunto de teste permaneceu isolado durante o ajuste.

A seleção dos hiperparâmetros foi orientada pelo MAE, considerando que essa métrica expressa diretamente, em quilogramas de leite, a magnitude média dos erros de previsão.

In [ ]:
# ============================================================
# 5.1 AJUSTE DE HIPERPARÂMETROS — GRADIENT BOOSTING C2
# ============================================================

from sklearn.model_selection import GridSearchCV


# ------------------------------------------------------------
# 1. CENÁRIO ESCOLHIDO
# ------------------------------------------------------------

cenario_gb = "C2_lactacao"

variaveis_gb = cenarios_ml[
    cenario_gb
]

X_gb = base_treino[
    variaveis_gb
].copy()

y_gb = base_treino[
    "kg_leite"
].copy()


# ------------------------------------------------------------
# 2. PIPELINE
# ------------------------------------------------------------

pipeline_gb = criar_pipeline_gradient_boosting(
    variaveis_gb
)


# ------------------------------------------------------------
# 3. GRADE DE HIPERPARÂMETROS
# ------------------------------------------------------------

grade_gb = {

    "modelo__n_estimators": [
        100,
        200,
        300
    ],

    "modelo__learning_rate": [
        0.03,
        0.05,
        0.10
    ],

    "modelo__max_depth": [
        2,
        3
    ],

    "modelo__min_samples_split": [
        2,
        5
    ],

    "modelo__min_samples_leaf": [
        1,
        3
    ]
}


# ------------------------------------------------------------
# 4. GRID SEARCH
# ------------------------------------------------------------

busca_gb = GridSearchCV(
    estimator=pipeline_gb,

    param_grid=grade_gb,

    scoring="neg_mean_absolute_error",

    cv=splits_cv,

    n_jobs=-1,

    refit=True,

    return_train_score=False
)


print("Iniciando ajuste do Gradient Boosting...")
print(
    "Número de combinações:",
    np.prod(
        [
            len(v)
            for v in grade_gb.values()
        ]
    )
)

busca_gb.fit(
    X_gb,
    y_gb
)


# ------------------------------------------------------------
# 5. MELHORES HIPERPARÂMETROS
# ------------------------------------------------------------

print("\nMELHORES HIPERPARÂMETROS")
print("-" * 60)

for parametro, valor in busca_gb.best_params_.items():
    print(
        parametro,
        "=",
        valor
    )


# ------------------------------------------------------------
# 6. MELHOR MAE DA BUSCA
# ------------------------------------------------------------

melhor_mae_gb = (
    -busca_gb.best_score_
)

print("\nMELHOR RESULTADO NA VALIDAÇÃO")
print("-" * 60)

print(
    "MAE médio:",
    round(
        melhor_mae_gb,
        3
    ),
    "kg"
)


# ------------------------------------------------------------
# 7. TOP 10 CONFIGURAÇÕES
# ------------------------------------------------------------

resultados_busca_gb = pd.DataFrame(
    busca_gb.cv_results_
)

top10_gb = (
    resultados_busca_gb
    .assign(
        MAE_medio=lambda df:
        -df["mean_test_score"]
    )
    .sort_values(
        "MAE_medio"
    )
    [
        [
            "rank_test_score",
            "MAE_medio",
            "std_test_score",
            "param_modelo__n_estimators",
            "param_modelo__learning_rate",
            "param_modelo__max_depth",
            "param_modelo__min_samples_split",
            "param_modelo__min_samples_leaf"
        ]
    ]
    .head(10)
    .reset_index(drop=True)
)

top10_gb["MAE_medio"] = (
    top10_gb["MAE_medio"]
    .round(3)
)

top10_gb["std_test_score"] = (
    top10_gb["std_test_score"]
    .round(3)
)


print("\nTOP 10 CONFIGURAÇÕES")
print("-" * 60)

display(top10_gb)

## 5.2 Resultado do ajuste do Gradient Boosting

O ajuste de hiperparâmetros do Gradient Boosting foi realizado exclusivamente sobre o conjunto de treinamento, utilizando os cinco folds agrupados por animal previamente definidos.

A melhor configuração utilizou 200 estimadores, taxa de aprendizado de 0,10, profundidade máxima igual a 2 e mínimo de três observações por folha.

O MAE médio obtido durante a busca foi de 3,780 kg. Antes da otimização, o Gradient Boosting no cenário C2 havia apresentado MAE médio de 3,799 kg.

Portanto, o ajuste proporcionou redução de aproximadamente 0,5% no erro absoluto médio. Embora o ganho tenha sido pequeno, a configuração otimizada apresentou também menor variação de desempenho entre os folds.

Esse resultado indica que a configuração inicial do Gradient Boosting já apresentava desempenho próximo das melhores combinações avaliadas.

### 5.3 Ajuste do Random Forest

O Random Forest apresentou seu melhor desempenho inicial no cenário C3, que inclui informações de propriedade, mês do controle, produção anterior, dias em lactação, idade e raça.

Embora seu desempenho inicial tenha sido inferior ao Gradient Boosting, o modelo foi submetido ao ajuste de hiperparâmetros para verificar se a otimização permitiria reduzir essa diferença.

Assim como no ajuste anterior, a busca foi realizada exclusivamente sobre o conjunto de treinamento, utilizando os mesmos cinco folds agrupados por animal e o MAE como critério de seleção.

O conjunto de teste permaneceu preservado e não participou da escolha dos hiperparâmetros.

In [ ]:
# ============================================================
# 5.3 AJUSTE DE HIPERPARÂMETROS — RANDOM FOREST C3
# ============================================================

# ------------------------------------------------------------
# 1. CENÁRIO ESCOLHIDO
# ------------------------------------------------------------

cenario_rf = "C3_completo_comum"

variaveis_rf = cenarios_ml[
    cenario_rf
]

X_rf = base_treino[
    variaveis_rf
].copy()

y_rf = base_treino[
    "kg_leite"
].copy()


# ------------------------------------------------------------
# 2. PIPELINE
# ------------------------------------------------------------

pipeline_rf = criar_pipeline_random_forest(
    variaveis_rf
)


# ------------------------------------------------------------
# 3. GRADE DE HIPERPARÂMETROS
# ------------------------------------------------------------

grade_rf = {

    "modelo__n_estimators": [
        300,
        500
    ],

    "modelo__max_depth": [
        None,
        8
    ],

    "modelo__min_samples_split": [
        2,
        5
    ],

    "modelo__min_samples_leaf": [
        1,
        2,
        3
    ],

    "modelo__max_features": [
        "sqrt",
        1.0
    ]
}


# ------------------------------------------------------------
# 4. GRID SEARCH
# ------------------------------------------------------------

busca_rf = GridSearchCV(
    estimator=pipeline_rf,

    param_grid=grade_rf,

    scoring="neg_mean_absolute_error",

    cv=splits_cv,

    n_jobs=-1,

    refit=True,

    return_train_score=False
)


print("Iniciando ajuste do Random Forest...")

print(
    "Número de combinações:",
    np.prod(
        [
            len(v)
            for v in grade_rf.values()
        ]
    )
)


busca_rf.fit(
    X_rf,
    y_rf
)


# ------------------------------------------------------------
# 5. MELHORES HIPERPARÂMETROS
# ------------------------------------------------------------

print("\nMELHORES HIPERPARÂMETROS")
print("-" * 60)

for parametro, valor in busca_rf.best_params_.items():

    print(
        parametro,
        "=",
        valor
    )


# ------------------------------------------------------------
# 6. MELHOR MAE
# ------------------------------------------------------------

melhor_mae_rf = (
    -busca_rf.best_score_
)

print("\nMELHOR RESULTADO NA VALIDAÇÃO")
print("-" * 60)

print(
    "MAE médio:",
    round(
        melhor_mae_rf,
        3
    ),
    "kg"
)


# ------------------------------------------------------------
# 7. TOP 10 CONFIGURAÇÕES
# ------------------------------------------------------------

resultados_busca_rf = pd.DataFrame(
    busca_rf.cv_results_
)

top10_rf = (
    resultados_busca_rf
    .assign(
        MAE_medio=lambda df:
        -df["mean_test_score"]
    )
    .sort_values(
        "MAE_medio"
    )
    [
        [
            "rank_test_score",
            "MAE_medio",
            "std_test_score",
            "param_modelo__n_estimators",
            "param_modelo__max_depth",
            "param_modelo__min_samples_split",
            "param_modelo__min_samples_leaf",
            "param_modelo__max_features"
        ]
    ]
    .head(10)
    .reset_index(drop=True)
)

top10_rf["MAE_medio"] = (
    top10_rf["MAE_medio"]
    .round(3)
)

top10_rf["std_test_score"] = (
    top10_rf["std_test_score"]
    .round(3)
)


print("\nTOP 10 CONFIGURAÇÕES")
print("-" * 60)

display(top10_rf)


# ------------------------------------------------------------
# 8. COMPARAÇÃO DOS DOIS MODELOS AJUSTADOS
# ------------------------------------------------------------

comparacao_ajustados = pd.DataFrame([
    {
        "modelo": "Gradient Boosting",
        "cenario": "C2_lactacao",
        "MAE_CV": melhor_mae_gb
    },
    {
        "modelo": "Random Forest",
        "cenario": "C3_completo_comum",
        "MAE_CV": melhor_mae_rf
    }
]).round(3)

print("\nMODELOS AJUSTADOS — COMPARAÇÃO")
print("-" * 60)

display(comparacao_ajustados)

### 5.4 Seleção do modelo final

Após o ajuste de hiperparâmetros, o Gradient Boosting permaneceu como o algoritmo de melhor desempenho na validação cruzada.

O Gradient Boosting ajustado no cenário C2 apresentou MAE médio de 3,780 kg, enquanto o Random Forest ajustado no cenário C3 apresentou MAE médio de 4,008 kg.

Dessa forma, o Gradient Boosting no cenário C2 foi selecionado como modelo final antes de qualquer avaliação sobre o conjunto de teste.

Essa decisão foi tomada exclusivamente com base nos resultados obtidos no conjunto de treinamento e na validação cruzada agrupada por animal.

O conjunto de teste, contendo animais que não participaram do treinamento ou da seleção de hiperparâmetros, foi utilizado após a seleção para estimar a capacidade de generalização do modelo final.

# 6. Avaliação final no conjunto de teste

O modelo selecionado foi avaliado sobre o conjunto de teste reservado, composto exclusivamente por animais que não participaram do treinamento.

Essa avaliação representa uma estimativa da capacidade do modelo de generalizar suas previsões para indivíduos não observados durante o ajuste.

Além das métricas globais, o desempenho foi analisado separadamente para cada propriedade, considerando as diferenças previamente identificadas entre os sistemas produtivos.

In [ ]:
# ============================================================
# 6.1 AVALIAÇÃO FINAL — GRADIENT BOOSTING
# ============================================================

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# ------------------------------------------------------------
# 1. MODELO FINAL
# ------------------------------------------------------------

# O GridSearchCV já reajustou automaticamente o melhor modelo
# utilizando todo o conjunto de treinamento.
modelo_final = busca_gb.best_estimator_


# ------------------------------------------------------------
# 2. DADOS DE TESTE
# ------------------------------------------------------------

X_teste_final = base_teste[
    variaveis_gb
].copy()

y_teste_final = base_teste[
    "kg_leite"
].copy()


# ------------------------------------------------------------
# 3. PREDIÇÕES
# ------------------------------------------------------------

predicoes_teste = modelo_final.predict(
    X_teste_final
)


# ------------------------------------------------------------
# 4. MÉTRICAS GLOBAIS
# ------------------------------------------------------------

mae_teste = mean_absolute_error(
    y_teste_final,
    predicoes_teste
)

rmse_teste = np.sqrt(
    mean_squared_error(
        y_teste_final,
        predicoes_teste
    )
)

r2_teste = r2_score(
    y_teste_final,
    predicoes_teste
)


print("AVALIAÇÃO FINAL — CONJUNTO DE TESTE")
print("-" * 60)

print(
    "Observações de teste:",
    len(base_teste)
)

print(
    "Animais de teste:",
    base_teste[
        "id_animal_global"
    ].nunique()
)

print(
    "MAE:",
    round(mae_teste, 3),
    "kg"
)

print(
    "RMSE:",
    round(rmse_teste, 3),
    "kg"
)

print(
    "R²:",
    round(r2_teste, 3)
)


# ------------------------------------------------------------
# 5. BASE COM RESULTADOS INDIVIDUAIS
# ------------------------------------------------------------

resultado_teste = base_teste[
    [
        "propriedade",
        "id_animal_global",
        "data_prova",
        "kg_leite"
    ]
].copy()

resultado_teste["kg_previsto"] = (
    predicoes_teste
)

resultado_teste["erro"] = (
    resultado_teste["kg_leite"]
    - resultado_teste["kg_previsto"]
)

resultado_teste["erro_absoluto"] = (
    resultado_teste["erro"].abs()
)


# ------------------------------------------------------------
# 6. DESEMPENHO POR PROPRIEDADE
# ------------------------------------------------------------

resultados_propriedade = []

for propriedade, grupo in resultado_teste.groupby(
    "propriedade"
):

    y_real = grupo[
        "kg_leite"
    ]

    y_pred = grupo[
        "kg_previsto"
    ]

    resultados_propriedade.append({
        "propriedade": propriedade,

        "observacoes": len(grupo),

        "animais": grupo[
            "id_animal_global"
        ].nunique(),

        "MAE": mean_absolute_error(
            y_real,
            y_pred
        ),

        "RMSE": np.sqrt(
            mean_squared_error(
                y_real,
                y_pred
            )
        ),

        "R2": r2_score(
            y_real,
            y_pred
        ),

        "erro_medio": (
            grupo["erro"].mean()
        )
    })


resultado_por_propriedade = (
    pd.DataFrame(
        resultados_propriedade
    )
    .round(3)
)


print("\nDESEMPENHO POR PROPRIEDADE")
print("-" * 60)

display(
    resultado_por_propriedade
)


# ------------------------------------------------------------
# 7. COMPARAÇÃO CV x TESTE
# ------------------------------------------------------------

comparacao_final = pd.DataFrame([
    {
        "etapa": "Validação cruzada",
        "MAE": melhor_mae_gb
    },
    {
        "etapa": "Teste final",
        "MAE": mae_teste
    }
]).round(3)

print("\nCOMPARAÇÃO — VALIDAÇÃO x TESTE")
print("-" * 60)

display(
    comparacao_final
)

## 6.2 Resultado final do modelo

O Gradient Boosting ajustado no cenário C2 foi avaliado no conjunto de teste reservado, composto por 189 controles referentes a 37 animais que não participaram do treinamento.

O modelo apresentou MAE de 3,685 kg, RMSE de 5,075 kg e R² de 0,749.

O desempenho observado no teste foi semelhante ao obtido durante a validação cruzada, na qual o MAE médio havia sido de 3,780 kg. Essa proximidade indica estabilidade do modelo quando aplicado a animais não utilizados durante o treinamento.

Em Cerqueira César, o modelo apresentou MAE de 3,796 kg, RMSE de 5,329 kg e R² de 0,647. O erro médio foi de -0,015 kg, indicando ausência de tendência sistemática relevante de superestimação ou subestimação nessa propriedade.

Na UENP, o MAE foi de 3,284 kg, o RMSE de 4,029 kg e o R² de 0,495. O erro médio foi de 1,078 kg, indicando tendência de subestimação da produção nessa propriedade. Entretanto, esse resultado deve ser interpretado considerando que o subconjunto de teste da UENP contém apenas 41 observações referentes a seis animais e apresenta menor amplitude de produção.

De forma geral, os resultados indicam que o modelo foi capaz de explicar aproximadamente 75% da variabilidade da produção de leite no conjunto de teste, mantendo erro absoluto médio inferior a 4 kg por controle.

In [ ]:
# ============================================================
# 6.3 BASELINE NO CONJUNTO DE TESTE
# ============================================================

media_treino = base_treino["kg_leite"].mean()

predicao_baseline_teste = np.full(
    len(base_teste),
    media_treino
)

mae_baseline_teste = mean_absolute_error(
    y_teste_final,
    predicao_baseline_teste
)

rmse_baseline_teste = np.sqrt(
    mean_squared_error(
        y_teste_final,
        predicao_baseline_teste
    )
)

r2_baseline_teste = r2_score(
    y_teste_final,
    predicao_baseline_teste
)

reducao_mae_percentual = (
    (mae_baseline_teste - mae_teste)
    / mae_baseline_teste
    * 100
)

comparacao_teste_final = pd.DataFrame([
    {
        "modelo": "Baseline — média do treino",
        "MAE": mae_baseline_teste,
        "RMSE": rmse_baseline_teste,
        "R2": r2_baseline_teste
    },
    {
        "modelo": "Gradient Boosting final",
        "MAE": mae_teste,
        "RMSE": rmse_teste,
        "R2": r2_teste
    }
]).round(3)

print("COMPARAÇÃO FINAL NO CONJUNTO DE TESTE")
print("-" * 60)

display(comparacao_teste_final)

print(
    "\nRedução do MAE em relação ao baseline:",
    f"{reducao_mae_percentual:.2f}%"
)

### 6.3 Comparação com o baseline no conjunto de teste

Para contextualizar o desempenho do modelo final, foi realizada uma comparação com um baseline que utiliza exclusivamente a média da produção observada no conjunto de treinamento.

No conjunto de teste, o baseline apresentou MAE de 8,444 kg, RMSE de 10,134 kg e R² aproximadamente igual a zero.

O Gradient Boosting final apresentou MAE de 3,685 kg, RMSE de 5,075 kg e R² de 0,749.

Em relação ao baseline, o modelo reduziu o erro absoluto médio em 56,36%.

O resultado demonstra que as informações utilizadas pelo modelo fornecem capacidade preditiva substancialmente superior à utilização isolada da média histórica da produção.

### 6.4 Análise dos erros de previsão

Além das métricas agregadas, foram analisados os erros individuais produzidos pelo modelo final.

Essa avaliação permite identificar a distribuição dos resíduos, a ocorrência de erros de maior magnitude e possíveis diferenças de desempenho relacionadas à disponibilidade das informações utilizadas como atributos.

Foi considerado como erro a diferença entre a produção observada e a produção prevista. Dessa forma, valores positivos indicam que o modelo subestimou a produção real, enquanto valores negativos indicam superestimação.

In [ ]:
# ============================================================
# 6.4 ANÁLISE DOS ERROS — MODELO FINAL
# ============================================================

# ------------------------------------------------------------
# 1. ADICIONAR AS VARIÁVEIS UTILIZADAS PELO MODELO
# ------------------------------------------------------------

resultado_teste["producao_anterior"] = (
    base_teste["producao_anterior"].to_numpy()
)

resultado_teste["producao_anterior_disponivel"] = (
    base_teste["producao_anterior_disponivel"].to_numpy()
)

resultado_teste["del_harmonizado"] = (
    base_teste["del_harmonizado"].to_numpy()
)

resultado_teste["del_disponivel"] = (
    base_teste["del_disponivel"].to_numpy()
)


# ------------------------------------------------------------
# 2. DISTRIBUIÇÃO DOS ERROS
# ------------------------------------------------------------

print("DISTRIBUIÇÃO DO ERRO")
print("-" * 60)

display(
    resultado_teste[
        "erro"
    ].describe(
        percentiles=[
            0.05,
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95
        ]
    ).round(3)
)


print("\nDISTRIBUIÇÃO DO ERRO ABSOLUTO")
print("-" * 60)

display(
    resultado_teste[
        "erro_absoluto"
    ].describe(
        percentiles=[
            0.50,
            0.75,
            0.90,
            0.95
        ]
    ).round(3)
)


# ------------------------------------------------------------
# 3. SUBESTIMAÇÃO E SUPERESTIMAÇÃO
# ------------------------------------------------------------

subestimacoes = (
    resultado_teste["erro"] > 0
).sum()

superestimacoes = (
    resultado_teste["erro"] < 0
).sum()

print("\nDIREÇÃO DOS ERROS")
print("-" * 60)

print(
    "Subestimações:",
    subestimacoes
)

print(
    "Superestimações:",
    superestimacoes
)

print(
    "Erro exatamente zero:",
    (resultado_teste["erro"] == 0).sum()
)


# ------------------------------------------------------------
# 4. MAIORES ERROS INDIVIDUAIS
# ------------------------------------------------------------

print("\n10 MAIORES ERROS ABSOLUTOS")
print("-" * 60)

maiores_erros = (
    resultado_teste
    .sort_values(
        "erro_absoluto",
        ascending=False
    )
    [
        [
            "propriedade",
            "id_animal_global",
            "data_prova",
            "kg_leite",
            "kg_previsto",
            "erro",
            "erro_absoluto",
            "producao_anterior",
            "del_harmonizado"
        ]
    ]
    .head(10)
)

display(
    maiores_erros.round(3)
)


# ------------------------------------------------------------
# 5. DESEMPENHO COM E SEM PRODUÇÃO ANTERIOR
# ------------------------------------------------------------

print("\nDESEMPENHO SEGUNDO DISPONIBILIDADE DA PRODUÇÃO ANTERIOR")
print("-" * 60)

resultado_prod_anterior = (
    resultado_teste
    .groupby(
        "producao_anterior_disponivel"
    )
    .agg(
        observacoes=("erro_absoluto", "size"),
        MAE=("erro_absoluto", "mean"),
        erro_medio=("erro", "mean")
    )
    .round(3)
)

display(
    resultado_prod_anterior
)


# ------------------------------------------------------------
# 6. DESEMPENHO COM E SEM DEL
# ------------------------------------------------------------

print("\nDESEMPENHO SEGUNDO DISPONIBILIDADE DO DEL")
print("-" * 60)

resultado_del = (
    resultado_teste
    .groupby(
        "del_disponivel"
    )
    .agg(
        observacoes=("erro_absoluto", "size"),
        MAE=("erro_absoluto", "mean"),
        erro_medio=("erro", "mean")
    )
    .round(3)
)

display(
    resultado_del
)

### 6.5 Interpretação dos erros de previsão

A análise individual das previsões demonstrou que o modelo apresenta baixa tendência sistemática de erro no conjunto de teste. O erro médio, definido como a diferença entre a produção observada e a produção prevista, foi de 0,222 kg.

Metade das previsões apresentou erro absoluto inferior a aproximadamente 2,97 kg, enquanto 75% dos controles apresentaram erro inferior a 4,84 kg. Em 90% das observações, o erro absoluto permaneceu abaixo de aproximadamente 8,73 kg.

Também foram identificadas observações com erros de maior magnitude, chegando a 20,43 kg no caso mais extremo. Esses registros demonstram que, embora o desempenho médio seja satisfatório, existem situações individuais nas quais a previsão pode apresentar desvios consideráveis.

A disponibilidade da produção anterior apresentou forte relação com a precisão das estimativas. Nos 157 controles em que essa informação estava disponível, o MAE foi de 3,150 kg. Nos 32 controles sem produção anterior válida, o MAE aumentou para 6,311 kg.

Esse resultado reforça a importância do histórico produtivo recente para a estimativa da produção atual.

Em relação aos dias em lactação, o MAE foi de 3,772 kg quando o atributo estava disponível e 3,204 kg quando estava ausente. Essa comparação não deve ser interpretada isoladamente como evidência de que a ausência do DEL melhora as previsões, pois os dois subconjuntos apresentam diferentes composições de animais e propriedades. Na comparação controlada entre cenários realizada durante a validação cruzada, a inclusão do DEL resultou em melhora do desempenho global.

# 7. Importância das variáveis

Após a avaliação do desempenho preditivo, foi analisada a contribuição das variáveis utilizadas pelo modelo final.

Foi empregada a técnica de importância por permutação. Nesse método, os valores de cada atributo são embaralhados individualmente, quebrando temporariamente sua relação com a variável-alvo.

A redução do desempenho provocada por essa perturbação permite estimar a contribuição preditiva de cada atributo.

A análise foi realizada após a seleção definitiva do modelo e possui finalidade interpretativa, não tendo sido utilizada para realizar novos ajustes ou selecionar outro algoritmo.

In [ ]:
# ============================================================
# 7.1 IMPORTÂNCIA POR PERMUTAÇÃO — MODELO FINAL
# ============================================================

from sklearn.inspection import permutation_importance


# ------------------------------------------------------------
# 1. CÁLCULO DA IMPORTÂNCIA
# ------------------------------------------------------------

importancia_permutacao = permutation_importance(
    modelo_final,
    X_teste_final,
    y_teste_final,

    scoring="neg_mean_absolute_error",

    n_repeats=30,

    random_state=SEMENTE,

    n_jobs=-1
)


# ------------------------------------------------------------
# 2. TABELA
# ------------------------------------------------------------

tabela_importancia = pd.DataFrame({
    "variavel": X_teste_final.columns,

    "aumento_MAE_medio": (
        importancia_permutacao[
            "importances_mean"
        ]
    ),

    "desvio": (
        importancia_permutacao[
            "importances_std"
        ]
    )
})


tabela_importancia = (
    tabela_importancia
    .sort_values(
        "aumento_MAE_medio",
        ascending=False
    )
    .reset_index(drop=True)
)


tabela_importancia[
    "aumento_MAE_medio"
] = (
    tabela_importancia[
        "aumento_MAE_medio"
    ].round(3)
)

tabela_importancia[
    "desvio"
] = (
    tabela_importancia[
        "desvio"
    ].round(3)
)


print("IMPORTÂNCIA DAS VARIÁVEIS — PERMUTAÇÃO")
print("-" * 60)

display(
    tabela_importancia
)

### 7.2 Interpretação da importância das variáveis

A análise por permutação indicou que a produção anterior foi o atributo de maior contribuição para o desempenho do modelo final.

Quando os valores dessa variável foram aleatoriamente permutados no conjunto de teste, o MAE aumentou, em média, 4,556 kg. Esse resultado reforça a importância do histórico produtivo recente para a estimativa da produção atual.

A propriedade de origem apresentou a segunda maior importância, com aumento médio de 1,418 kg no MAE após a permutação. Esse resultado é compatível com as diferenças produtivas identificadas entre as duas propriedades durante a análise exploratória.

Os dias em lactação também apresentaram contribuição relevante, provocando aumento médio de 1,349 kg no MAE quando sua relação com as demais observações foi perturbada.

O mês do controle apresentou contribuição menor, porém ainda mensurável, com aumento médio de 0,605 kg no erro.

O indicador de disponibilidade da produção anterior apresentou importância de 0,378 kg, demonstrando que a ausência dessa informação também contém informação útil para o modelo.

Por outro lado, o indicador de disponibilidade dos dias em lactação apresentou contribuição praticamente nula.

Os valores de importância por permutação representam contribuição preditiva e não devem ser interpretados como relações causais entre os atributos e a produção de leite.

A relevância da variável de propriedade também evidencia que o modelo foi validado para generalização entre animais não observados das duas propriedades analisadas. Sua aplicação direta a propriedades externas exigiria nova validação com dados provenientes desses sistemas produtivos.

# 8. Persistência do modelo e dos resultados

Após a avaliação final, o pipeline correspondente ao modelo selecionado foi armazenado para permitir sua reutilização nas etapas posteriores do sistema.

Foi preservado o modelo treinado exclusivamente sobre o conjunto de treinamento e utilizado na avaliação final, mantendo correspondência exata entre o modelo salvo e os resultados reportados.

Também foram exportadas as previsões obtidas no conjunto de teste e a importância das variáveis.

A versão destinada à implantação constitui uma etapa posterior à avaliação experimental concluída e ainda não foi gerada neste notebook.

In [ ]:
# ============================================================
# 8.1 SALVAMENTO DO MODELO E DOS RESULTADOS
# ============================================================

from pathlib import Path
import joblib
import json


# ------------------------------------------------------------
# 1. DIRETÓRIOS
# ------------------------------------------------------------

pasta_modelos = Path("../modelos")
pasta_resultados = Path("../resultados")

pasta_modelos.mkdir(
    parents=True,
    exist_ok=True
)

pasta_resultados.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# 2. MODELO AVALIADO
# ------------------------------------------------------------

arquivo_modelo = (
    pasta_modelos
    / "gradient_boosting_c2_modelo_avaliado.joblib"
)

joblib.dump(
    modelo_final,
    arquivo_modelo
)


# ------------------------------------------------------------
# 3. PREVISÕES DO TESTE
# ------------------------------------------------------------

arquivo_predicoes = (
    pasta_resultados
    / "predicoes_teste_modelo_final.csv"
)

resultado_teste.to_csv(
    arquivo_predicoes,
    index=False,
    encoding="utf-8-sig"
)


# ------------------------------------------------------------
# 4. IMPORTÂNCIA DAS VARIÁVEIS
# ------------------------------------------------------------

arquivo_importancia = (
    pasta_resultados
    / "importancia_variaveis_modelo_final.csv"
)

tabela_importancia.to_csv(
    arquivo_importancia,
    index=False,
    encoding="utf-8-sig"
)


# ------------------------------------------------------------
# 5. METADADOS
# ------------------------------------------------------------

metadados_modelo = {
    "algoritmo": "GradientBoostingRegressor",
    "cenario": "C2_lactacao",

    "variaveis": variaveis_gb,

    "hiperparametros": {
        chave.replace("modelo__", ""): valor
        for chave, valor
        in busca_gb.best_params_.items()
    },

    "validacao_cruzada": {
        "tipo": "StratifiedGroupKFold",
        "folds": 5,
        "agrupamento": "id_animal_global",
        "estratificacao": "propriedade",
        "MAE_medio": round(
            float(melhor_mae_gb),
            3
        )
    },

    "teste_final": {
        "observacoes": int(
            len(base_teste)
        ),
        "animais": int(
            base_teste[
                "id_animal_global"
            ].nunique()
        ),
        "MAE": round(
            float(mae_teste),
            3
        ),
        "RMSE": round(
            float(rmse_teste),
            3
        ),
        "R2": round(
            float(r2_teste),
            3
        ),
        "reducao_MAE_baseline_percentual": round(
            float(reducao_mae_percentual),
            2
        )
    }
}


arquivo_metadados = (
    pasta_modelos
    / "gradient_boosting_c2_metadados.json"
)

with open(
    arquivo_metadados,
    "w",
    encoding="utf-8"
) as arquivo:

    json.dump(
        metadados_modelo,
        arquivo,
        ensure_ascii=False,
        indent=4
    )


# ------------------------------------------------------------
# 6. CONFIRMAÇÃO
# ------------------------------------------------------------

print("ARQUIVOS SALVOS")
print("-" * 60)

print(
    "Modelo:",
    arquivo_modelo
)

print(
    "Metadados:",
    arquivo_metadados
)

print(
    "Predições de teste:",
    arquivo_predicoes
)

print(
    "Importância das variáveis:",
    arquivo_importancia
)